In [41]:
import pandas as pd
import numpy as np 
import math
import matplotlib.pyplot as plt
import joblib
import os 

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, recall_score


from surprise import Dataset, Reader , accuracy
from surprise.prediction_algorithms.matrix_factorization import SVD
from surprise.model_selection import GridSearchCV


from sklearn.model_selection import train_test_split 
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
moviesDF = pd.read_csv("./ml-1m/movies.dat" , delimiter="::", encoding="latin-1" ,names=["MovieID", "Title", "Genres"])
ratingsDF = pd.read_csv("./ml-1m/ratings.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "MovieID", "Rating", "Timestamp"])
usersDF = pd.read_csv("./ml-1m/users.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "Gender", "Age", "Occupation", "Zip-code"])

C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_18148\336456026.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  moviesDF = pd.read_csv("./ml-1m/movies.dat" , delimiter="::", encoding="latin-1" ,names=["MovieID", "Title", "Genres"])
C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_18148\336456026.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratingsDF = pd.read_csv("./ml-1m/ratings.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "MovieID", "Rating", "Timestamp"])
C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_18148\336456026.py:3: ParserWarning: Falling back to the 'python' en

In [3]:
print("movies cols:", moviesDF.columns) 
print("ratings cols:", ratingsDF.columns)
print("user DF:", usersDF.columns)

print(moviesDF.shape)
print(ratingsDF.shape)
print(usersDF.shape)

movies cols: Index(['MovieID', 'Title', 'Genres'], dtype='object')
ratings cols: Index(['UserID', 'MovieID', 'Rating', 'Timestamp'], dtype='object')
user DF: Index(['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'], dtype='object')
(3883, 3)
(1000209, 4)
(6040, 5)


<h3><b>Separate the non-rated and rated movies </b></h3>

In [4]:
rated_movies = ratingsDF.merge(moviesDF, on="MovieID")

rated_movies = moviesDF.loc[moviesDF["MovieID"].isin(rated_movies["MovieID"].unique())] 
print("rated movies shape: ",rated_movies.shape)

non_rated_movies = moviesDF.loc[~moviesDF["MovieID"].isin(rated_movies["MovieID"].unique())] 
print("Non rated movies shape: ", non_rated_movies.shape)

rated_movies["Title"] = rated_movies["Title"].str.lower()

non_rated_movies.head()

rated movies shape:  (3706, 3)
Non rated movies shape:  (177, 3)


C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_18148\1260981810.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rated_movies["Title"] = rated_movies["Title"].str.lower()


,MovieID,Title,Genres
50,51,Guardian Angel (1994),Action|Drama|Thriller
107,109,Headless Body in Topless Bar (1995),Comedy
113,115,Happiness Is in the Field (1995),Comedy
141,143,Gospa (1995),Drama
281,284,New York Cop (1996),Action|Crime


<h3><b>Check the sparsity: how many movies not yet rated by all user in percentage</b></h3>

In [5]:
n_unique_user = ratingsDF["UserID"].unique().shape
n_unique_movies = ratingsDF["MovieID"].unique().shape

print(n_unique_user, n_unique_movies)

spaecity = round(1.0 - len(ratingsDF)/float(n_unique_user[0]*n_unique_movies[0]) , 3)# round in 3 decimal point
print(f"The Sparcity level is : {spaecity*100}%")

(6040,) (3706,)
The Sparcity level is : 95.5%


<h2><b>Now apply SVD from Surprise</b></h2>

In [6]:
svd_merged_movie_rating = ratingsDF.merge(moviesDF, on="MovieID", how="left")
# svd_merged_movie_rating["Rating"] = svd_merged_movie_rating["Rating"]/max(svd_merged_movie_rating["Rating"])

print(svd_merged_movie_rating.shape)
print(max(svd_merged_movie_rating["UserID"]))
svd_merged_movie_rating.head()

(1000209, 6)
6040


,UserID,MovieID,Rating,Timestamp,Title,Genres
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy


In [7]:
def splitter():
    svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df = train_test_split(svd_merged_movie_rating, test_size = 0.001)
    return (svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df)
  
while True: 
    svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df = splitter()
    if sorted(svd_merged_movie_rating_train_df["UserID"].unique()) == [x for x in range(1, 6041)]: 
        break

In [8]:
# create and apply reader in train dataaset
reader = Reader(
    rating_scale=(
        min(svd_merged_movie_rating["Rating"].to_numpy()),
        max(svd_merged_movie_rating["Rating"].to_numpy())
    )
)

svd_reader_train = Dataset.load_from_df(svd_merged_movie_rating_train_df[['UserID', 'MovieID', 'Rating']], reader)
svd_reader_trainset = svd_reader_train.build_full_trainset() # this will use while training with "SVD" not for Hyper-parameter-tuner

<h4><b>Apply hyper parameter tuning on SVD usnig "GridSearchCV"</b></h4>

In [9]:
hyperParamGrids = {
    "n_factors": [50, 100, 150, 200],
    "lr_all" : [0.002, 0.003, 0.005, 0.007, 0.009, 0.01, 0.02, 0.05],
    "reg_all": [0.002, 0.003, 0.005, 0.007, 0.009, 0.01, 0.02, 0.05]
}

gridSearch = GridSearchCV(
    algo_class = SVD, 
    param_grid = hyperParamGrids, 
    measures=["rmse", "mae"], 
    cv= 3, 
    refit=False , 
    joblib_verbose = 2,
    n_jobs=-1, 
)

gridSearch.fit(svd_reader_train)


print("Best RMSE score:", gridSearch.best_score['rmse'])
print("Best MAE score:", gridSearch.best_score['mae'])

print()
print("Best hyperparameters in RMSE:")
print("n_factors:", gridSearch.best_params['rmse']['n_factors'])
print("lr_all:", gridSearch.best_params['rmse']['lr_all'])
print("reg_all:", gridSearch.best_params['rmse']['reg_all'])


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:   30.5s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 341 tasks      | elapsed:  9.5min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed: 18.7min


Best RMSE score: 0.8641429034628162
Best MAE score: 0.6809474146522486

Best hyperparameters in RMSE:
n_factors: 100
lr_all: 0.01
reg_all: 0.05


[Parallel(n_jobs=-1)]: Done 768 out of 768 | elapsed: 24.2min finished


In [10]:
model = SVD(
    n_factors= gridSearch.best_params['rmse']['n_factors'], 
    n_epochs= 200,
    biased= True,
    lr_all = gridSearch.best_params['rmse']['lr_all'],
    reg_all= gridSearch.best_params['rmse']['reg_all'], #
    verbose = True
)
model.fit(svd_reader_trainset)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49
Processing

In [11]:
prediction = model.predict(uid=5050, iid=51)
print(prediction.est)

3.7333299414940497


In [12]:
# now validate the model 

testlist = list(zip(svd_merged_movie_rating_test_df["UserID"], svd_merged_movie_rating_test_df["MovieID"], svd_merged_movie_rating_test_df["Rating"])) 
# it is list of tuples: [(UserID, MovieID, Rating), (UserID, MovieID, Rating),(UserID, MovieID, Rating).......]


pred2 = model.test(testlist)
print(accuracy.mae(pred2))
print(accuracy.mse(pred2))
print(accuracy.rmse(pred2))

MAE:  0.6594
0.6593800198320554
MSE: 0.6794
0.679390929507211
RMSE: 0.8243
0.8242517391593487


<h5><b> Now get the pu-> user latent vector and qi-> item latent vector</b></h5>

In [39]:
user_factors = model.pu 
item_factors = model.qi

print(f"User latent vector: {user_factors.shape}")
print(f"Item latent vector: {item_factors.shape}")

User latent vector: (6040, 100)
Item latent vector: (3706, 100)


<h7><b>as SVD pu and qi organized the user and differnt order so I need organize them manually</b></h7>

In [14]:
svd_userBias = []
svd_itemBias = []

user_vector = []
item_vector = []

# fill svd_userBias and user_vector
for x in sorted(svd_merged_movie_rating["UserID"].unique(), reverse=False): 
    user_inner_id = svd_reader_trainset.to_inner_uid(x)
    
    svd_userBias.append(model.bu[user_inner_id])
    user_vector.append(model.pu[user_inner_id])
    
    
# fill svd_itemBias and item_vector
for x in sorted(svd_merged_movie_rating["MovieID"].unique(), reverse=False): 
    movie_inner_id = svd_reader_trainset.to_inner_iid(x)
    
    svd_itemBias.append(model.bi[movie_inner_id])
    item_vector.append(model.qi[movie_inner_id])
    
    
    
svd_userBias = np.array(svd_userBias)
svd_itemBias = np.array(svd_itemBias)

user_vector = np.array(user_vector)
item_vector = np.array(item_vector)

In [15]:
SVD_Final_Vector_table = user_vector @ item_vector.T + svd_userBias.reshape(-1, 1) + svd_itemBias.reshape(1, -1) + svd_reader_trainset.global_mean

SVD_rated_Vector_table = pd.DataFrame(SVD_Final_Vector_table , columns=sorted(svd_merged_movie_rating["MovieID"].unique(), reverse=False)) 

SVD_non_rated_Vector_table = np.full((6040, len(non_rated_movies["MovieID"].unique())), svd_reader_trainset.global_mean)
SVD_non_rated_Vector_table = pd.DataFrame(SVD_non_rated_Vector_table, columns=non_rated_movies["MovieID"].unique())

SVD_Final_Vector_table = pd.concat([SVD_rated_Vector_table, SVD_non_rated_Vector_table], axis=1)
SVD_Final_Vector_table = SVD_Final_Vector_table[sorted(SVD_Final_Vector_table.columns)]

print(SVD_Final_Vector_table.shape)
SVD_Final_Vector_table

(6040, 3883)


,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,4.647578,3.551059,3.805659,3.184686,3.491043,3.851992,3.945386,3.717861,2.688253,3.801458,...,2.486167,3.446387,2.154114,2.076458,3.562867,4.096827,3.619644,4.547388,3.900354,3.710927
1,4.152664,3.232915,3.337719,3.047278,3.393266,3.533164,3.517040,3.345721,2.716464,3.266084,...,2.556213,2.912170,2.149583,2.252888,3.235254,3.178354,3.082575,3.989142,3.890823,3.673627
2,3.910400,3.684677,3.642851,2.239654,3.182566,3.996465,4.052878,3.369629,2.990206,3.907494,...,3.463113,3.216367,2.165985,2.407686,3.093302,3.435108,3.399053,3.501200,4.176466,3.655587
3,4.633827,3.310426,3.645271,2.428127,3.274639,4.184561,3.792220,3.684958,2.777163,3.700523,...,3.241577,3.684931,2.264790,2.601517,3.700837,3.783699,4.376187,3.622061,4.339043,3.955423
4,3.730145,2.371643,2.285134,2.029989,1.387237,2.580758,1.967319,2.840892,1.639258,2.547446,...,3.584585,1.894629,0.954368,0.923102,3.714397,2.745747,3.732291,3.239411,2.753163,2.772895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6035,3.962291,2.502007,2.540014,2.402519,2.334021,3.468806,2.664071,2.505737,1.946573,2.793899,...,2.890785,2.051661,1.178702,1.757970,3.470500,3.159332,4.260824,3.793502,3.655358,3.598327
6036,3.875134,3.114861,3.003445,2.918733,2.948915,3.519902,2.902193,2.886695,2.741901,3.108446,...,2.534985,2.958366,1.767013,2.473698,3.289922,3.628516,3.410987,3.737710,4.067139,3.594405
6037,3.990058,3.467656,2.746715,3.257173,2.925293,3.641340,3.572402,2.924656,3.022532,3.436058,...,3.359279,2.958554,1.430176,1.882267,3.308780,3.715451,3.510311,3.582785,3.752649,4.283313
6038,4.036085,3.263847,3.398557,2.985876,3.282793,3.551529,3.492905,3.099905,2.696681,3.675141,...,3.600016,3.292746,1.720388,2.563354,3.380564,3.648455,4.100536,3.725521,3.807759,4.307852


In [16]:
table = svd_merged_movie_rating_test_df.copy()

actual_rating = []
predicted_rating = []


for uid in sorted(table["UserID"].unique()): 
    userData = table.loc[table["UserID"] == uid].sort_values(by="MovieID", ascending=True)
    user_MovieId = userData["MovieID"]
    
    user_Actaual_Rating = list(userData["Rating"].to_numpy())    
    
    model_pred = []
    for mid in user_MovieId: 
        model_pred.append(SVD_Final_Vector_table.loc[uid-1][mid])
    
    actual_rating.extend(user_Actaual_Rating)
    predicted_rating.extend(model_pred)
    
print(len(actual_rating))
print(len(predicted_rating))
    

1001
1001


In [53]:
mseii = mean_squared_error(actual_rating, predicted_rating)
rmseii = np.sqrt(mseii)
print(f"mse:{mseii/5.00:0.3f}  ||  RMSE:{rmseii/5.00:0.3f}")

mse:0.136  ||  RMSE:0.165


In [54]:
maeii = mean_absolute_error(actual_rating, predicted_rating)
print(f"SVD model MAE: {maeii/5.00:.4f}")

SVD model MAE: 0.1319


<h2><b> Start work with Content(TF-IDF) filtering ========</b></h2>

<h5>Now build TF-IDF tables for rated and non-rated movies: =====================</h5>

In [18]:
prep_all_movies = ColumnTransformer(
    transformers=[
        ('genresTFIDF',  TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(1,18),
            max_df=0.9,
            min_df=1,
            max_features=500,
            analyzer="word", 
            token_pattern=r'[^|]+'
        ), 'Genres')
    ],
    remainder='drop'
)

prep_all_movies.fit(moviesDF)

ColumnTransformer(transformers=[('genresTFIDF',
                                 TfidfVectorizer(max_df=0.9, max_features=500,
                                                 ngram_range=(1, 18),
                                                 stop_words='english',
                                                 token_pattern='[^|]+'),
                                 'Genres')])

<h5>Now build TF-IDF tables for rated movies: ====================== </h5>

* main goal here is: to create a TF-IDF table to expalain - how the rated movies(3706 number of movies) matches to the rated-movies(3706 number of movies)

In [19]:
rated_tfidfTable = prep_all_movies.transform(rated_movies)

rated_cosineSim = cosine_similarity(rated_tfidfTable, rated_tfidfTable) # value range: 0-1
print(rated_cosineSim.shape) # (number of rated movies , number of rated movies)


rated_tfidf_df =pd.DataFrame(rated_cosineSim, columns=rated_movies["MovieID"].to_numpy()) # shape -> (number of user , number of rated movies)
print(rated_tfidf_df.shape) 


(3706, 3706)
(3706, 3706)


<h5>Now build TF-IDF tables for non-rated movies: =========================</h5>

* main goal here is: to create a TF-IDF table to expalain - how the non rated movies(177 movies) matches to the rated-movies(3706 number of movies)

In [20]:
non_rated_tfidfTable = prep_all_movies.transform(non_rated_movies)
print(non_rated_tfidfTable.shape) 


non_rated_cosineSim = cosine_similarity(non_rated_tfidfTable, rated_tfidfTable).T
print(non_rated_cosineSim.shape) # (number of non-rated movies , number of rated movies)

non_rated_tfidf_df = pd.DataFrame(non_rated_cosineSim, columns=non_rated_movies["MovieID"].to_numpy()) # shape -> (number of user , number of rated movies)
print(non_rated_tfidf_df.shape) 

(177, 398)
(3706, 177)
(3706, 177)


<h4><b> Now combine rated TF-IDF and non rated tf-idf table =====</b></h4>

In [21]:
combinedPivoted = MinMaxScaler(feature_range=(0, 5))

combined_tfidf_df = pd.concat([rated_tfidf_df, non_rated_tfidf_df] , axis=1)
combined_tfidf_df = combined_tfidf_df[sorted(combined_tfidf_df.columns)]
combined_tfidf_df_t = combinedPivoted.fit_transform(combined_tfidf_df)
combined_tfidf_df = pd.DataFrame(combined_tfidf_df_t , columns=combined_tfidf_df.columns)

combined_tfidf_df

,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,5.000000,0.500009,0.371881,0.435882,0.970815,0.000000,0.371881,0.866033,0.0,0.000000,...,0.970815,0.435882,2.042611,0.000000,0.000000,0.970815,0.000000,0.000000,0.000000,0.000000
1,0.500009,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.886777,0.0,0.441753,...,0.000000,0.000000,0.849529,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.371881,0.000000,5.000000,0.859944,1.915303,0.000000,5.000000,0.000000,0.0,0.000000,...,1.915303,0.859944,0.000000,0.000000,0.000000,1.915303,0.000000,0.000000,0.000000,0.000000
3,0.435882,0.000000,0.859944,5.000000,2.244929,0.000000,0.859944,0.000000,0.0,0.000000,...,2.244929,5.000000,0.000000,0.341380,0.000000,2.244929,1.946104,1.946104,1.946104,0.616056
4,0.970815,0.000000,1.915303,2.244929,5.000000,0.000000,1.915303,0.000000,0.0,0.000000,...,5.000000,2.244929,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3701,0.970815,0.000000,1.915303,2.244929,5.000000,0.000000,1.915303,0.000000,0.0,0.000000,...,5.000000,2.244929,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000
3702,0.000000,0.000000,0.000000,1.946104,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,1.946104,0.000000,0.877085,0.000000,0.000000,5.000000,5.000000,5.000000,1.582794
3703,0.000000,0.000000,0.000000,1.946104,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,1.946104,0.000000,0.877085,0.000000,0.000000,5.000000,5.000000,5.000000,1.582794
3704,0.000000,0.000000,0.000000,1.946104,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,1.946104,0.000000,0.877085,0.000000,0.000000,5.000000,5.000000,5.000000,1.582794


In [22]:
ratingsDFPivoted = ratingsDF.pivot(index="UserID", columns="MovieID", values="Rating").fillna(0)
ratingsDFPivoted = (ratingsDFPivoted != 0).astype(int)
print(ratingsDFPivoted.shape, combined_tfidf_df.shape)

comb_svd_tfIdf3 = pd.DataFrame(np.dot(ratingsDFPivoted, combined_tfidf_df), columns=combined_tfidf_df.columns)

print(type(comb_svd_tfIdf3))
print(comb_svd_tfIdf3.shape)
comb_svd_tfIdf3

(6040, 3706) (3706, 3883)
<class 'pandas.core.frame.DataFrame'>
(6040, 3883)


,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,58.434943,14.227358,18.022821,42.266930,26.401919,5.192940,18.022821,20.693129,5.564779,6.668476,...,26.401919,42.266930,40.068411,17.944151,7.490310,26.401919,69.827340,69.827340,69.827340,27.690178
1,9.970392,7.826599,53.653158,130.513299,51.350637,49.382684,53.653158,12.597875,91.379713,60.297627,...,51.350637,130.513299,7.101587,99.518364,51.341501,51.350637,203.561339,203.561339,203.561339,103.287617
2,26.961315,14.801039,36.660546,47.636674,77.004429,9.712740,36.660546,23.291363,30.922767,44.120953,...,77.004429,47.636674,17.009598,15.307971,8.441731,77.004429,13.591844,13.591844,13.591844,11.822106
3,0.314911,3.751049,0.964409,3.801503,0.000000,10.798494,0.964409,4.650093,32.275964,16.039469,...,0.000000,3.801503,2.621318,17.303229,5.843573,0.000000,9.766961,9.766961,9.766961,6.099782
4,48.004501,7.907206,119.634744,239.227562,166.989626,64.563762,119.634744,13.695560,59.379862,47.555654,...,166.989626,239.227562,13.554475,114.507902,80.496023,166.989626,305.862464,305.862464,305.862464,159.033460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6035,252.589453,103.499800,506.944432,929.452782,699.389959,202.224461,506.944432,146.550325,246.540303,228.033133,...,699.389959,929.452782,125.820721,407.651865,318.787725,699.389959,1134.015227,1134.015227,1134.015227,572.535873
6036,37.339813,10.890599,91.147173,224.928540,141.861237,69.183346,91.147173,12.194546,37.833003,56.552505,...,141.861237,224.928540,7.381259,115.809270,157.821402,141.861237,261.586511,261.586511,261.586511,185.959821
6037,11.490539,1.081039,18.189522,31.139689,29.562616,0.622085,18.189522,1.872397,2.525153,1.185278,...,29.562616,31.139689,3.881337,4.393921,0.000000,29.562616,20.971925,20.971925,20.971925,6.638849
6038,57.866316,18.072117,104.690967,106.755236,134.589049,9.729406,104.690967,23.132018,9.667096,15.023157,...,134.589049,106.755236,26.723026,21.186209,21.284368,134.589049,49.819088,49.819088,49.819088,29.900874


<h3><b> Conver the tables to dataframe: =========== </b></h3>

In [23]:
# SVD_Final_Vector_table conversion
SVD_Final_Vector_temp = SVD_Final_Vector_table.reset_index()
SVD_Final_Vector_df = SVD_Final_Vector_temp.melt(
    id_vars='index', 
    var_name='MovieID',
    value_name='SVD_Rating'
)
SVD_Final_Vector_df = SVD_Final_Vector_df.rename(columns={'index': 'UserID'})
SVD_Final_Vector_df["UserID"] +=1

SVD_Final_Vector_df

,UserID,MovieID,SVD_Rating
0,1,1,4.647578
1,2,1,4.152664
2,3,1,3.910400
3,4,1,4.633827
4,5,1,3.730145
...,...,...,...
23453315,6036,3952,3.598327
23453316,6037,3952,3.594405
23453317,6038,3952,4.283313
23453318,6039,3952,4.307852


In [24]:
# comb_svd_tfIdf3
comb_svd_tfIdf_temp = comb_svd_tfIdf3.reset_index()
comb_svd_tfIdf_df = comb_svd_tfIdf_temp.melt(
    id_vars='index', 
    var_name='MovieID',
    value_name='TfIdf_Rating'
)
comb_tfIdf_df = comb_svd_tfIdf_df.rename(columns={'index': 'UserID'})
comb_tfIdf_df["UserID"] +=1

comb_tfIdf_df

,UserID,MovieID,TfIdf_Rating
0,1,1,58.434943
1,2,1,9.970392
2,3,1,26.961315
3,4,1,0.314911
4,5,1,48.004501
...,...,...,...
23453315,6036,3952,572.535873
23453316,6037,3952,185.959821
23453317,6038,3952,6.638849
23453318,6039,3952,29.900874


<h3><b> Now Train Gradient boosting to see the result</b></h3>

In [25]:
comb_df = comb_tfIdf_df.merge(SVD_Final_Vector_df, on=["UserID", "MovieID"], how="left")
comb_df= comb_df.merge(ratingsDF, on=["UserID", "MovieID"], how="left").drop(["Timestamp"], axis=1)

combinedScale1 = MinMaxScaler(feature_range=(0, 5))
comb_df["TfIdf_Rating"] = combinedScale1.fit_transform(comb_df[["TfIdf_Rating"]])

comb_cleaned_df = comb_df.dropna()
comb_cleaned_df

,UserID,MovieID,TfIdf_Rating,SVD_Rating,Rating
0,1,1,0.086431,4.647578,5.0
5,6,1,0.064395,4.170604,4.0
7,8,1,0.025088,4.156822,4.0
8,9,1,0.059664,4.285008,5.0
9,10,1,0.320066,4.583211,5.0
...,...,...,...,...,...
23453091,5812,3952,0.814471,4.220027,4.0
23453110,5831,3952,0.985753,4.021482,3.0
23453116,5837,3952,0.458087,3.917131,4.0
23453206,5927,3952,0.265459,2.536974,1.0


In [26]:
gradient_x = comb_cleaned_df.iloc[:, 2:4]
gradient_y = comb_cleaned_df.iloc[:, -1]
gradient_train_x, gradient_test_x, gradient_train_y, gradient_test_y = train_test_split(gradient_x, gradient_y, test_size=0.001)

gradient_model = GradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.01, 
    n_estimators= 700, 
    subsample= 0.90, 
    max_depth=7,  
    random_state=42
)
gradient_model.fit(gradient_train_x, gradient_train_y)

GradientBoostingRegressor(learning_rate=0.01, max_depth=7, n_estimators=700,
                          random_state=42, subsample=0.9)

In [27]:
dummyPred = gradient_model.predict([[0.086431, 4.241818]])
print(dummyPred)

[4.45330706]


C:\Users\niaz mahmud\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [45]:
gradient_pred = gradient_model.predict(gradient_test_x)
gradien_rmse = mean_squared_error(gradient_test_y, gradient_pred)
print(f"gradien boosts model RMSE: {gradien_rmse/5.00:.4f}")

gradien boosts model RMSE: 0.0586


In [47]:
GBRmae = mean_absolute_error(gradient_test_y, gradient_pred)
print(f"radien boosts model MAE: {GBRmae/5.00:.4f}")

radien boosts model MAE: 0.0782


<h3><b> Get top N unrated movies from SVD and Tf-Idf array</b></h3>

In [30]:
def getTopN_Movie(SVDArray, Tf_IdfArray, combined_tfidf_dfs, ratingsDF, UID, movieID, ret_n_movie): 
    rated_mov_by_uid = ratingsDF.loc[ratingsDF["UserID"]==UID]
    unrated_mov_by_uid = ratingsDF.loc[~ratingsDF["MovieID"].isin(rated_mov_by_uid["MovieID"])].drop(["Timestamp"], axis=1)["MovieID"].unique()    
    unrated_mov_by_uid = pd.DataFrame({
        "MovieID": unrated_mov_by_uid
    })
    
    
    svd_user_all_rated_movie = SVDArray.loc[UID-1].to_numpy()
    tfIdf_all_movies = Tf_IdfArray.loc[UID-1].to_numpy()
    svd_user_all_rated_movie= pd.DataFrame({
        "MovieID": [x for x in range(1, len(svd_user_all_rated_movie)+1)], 
        "Rating": svd_user_all_rated_movie
    })
    tfIdf_all_movies = pd.DataFrame({
        "MovieID": [x for x in range(1, len(tfIdf_all_movies)+1)], 
        "Rating": tfIdf_all_movies
    })

    
    pure_tfIdf = combined_tfidf_dfs.loc[movieID].to_numpy()

    pure_tfIdf = pd.DataFrame({
        "MovieID": combined_tfidf_dfs.columns, 
        "Pure_TfIdf_Rating": pure_tfIdf
    })
    print(combined_tfidf_dfs.columns)
    
    merged_all = unrated_mov_by_uid.merge(svd_user_all_rated_movie , on=["MovieID"])
    merged_all = merged_all.merge(tfIdf_all_movies, on=["MovieID"])
    merged_all = merged_all.merge(pure_tfIdf, on=["MovieID"])
    merged_all = merged_all.rename(
        columns={"Rating_x": "SVD_Rating" , "Rating_y":"TfIdf_Rating"}
    )
    
    combinedScale = MinMaxScaler(feature_range=(0, 5))
    merged_all["TfIdf_Rating"] = combinedScale.fit_transform(merged_all[["TfIdf_Rating"]])
    
    if 3706-len(unrated_mov_by_uid)>30: 
        # if user rated less that 30 movies then we mostly relay on Tf-IDF
        merged_all["Combined_Ratinng"] =  (0.2*merged_all["SVD_Rating"]) + (0.6*merged_all["TfIdf_Rating"]) + (0.2*merged_all["Pure_TfIdf_Rating"])
    else: 
        # now user has enough rated movies ... we ralay in mostly SVD
        merged_all["Combined_Ratinng"] =  (0.4*merged_all["SVD_Rating"]) + (0.2*merged_all["TfIdf_Rating"])+ (0.4*merged_all["Pure_TfIdf_Rating"])

        
    merged_all = merged_all.sort_values(["Combined_Ratinng"], ascending=False)
    
    return merged_all.iloc[0:ret_n_movie, :]

    
    
        
    
topN_unrated_movies = getTopN_Movie(SVD_Final_Vector_table, comb_svd_tfIdf3, combined_tfidf_df, ratingsDF, UID=4, movieID=50 , ret_n_movie=50)
topN_unrated_movies


Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
       3943, 3944, 3945, 3946, 3947, 3948, 3949, 3950, 3951, 3952],
      dtype='int64', length=3883)


,MovieID,SVD_Rating,TfIdf_Rating,Pure_TfIdf_Rating,Combined_Ratinng
3517,3242,4.912373,5.000000,5.0,4.964949
3494,1886,4.689428,4.246514,5.0,4.725074
3302,2923,4.042122,5.000000,5.0,4.616849
785,2335,3.838200,5.000000,5.0,4.535280
1584,3473,3.581583,5.000000,5.0,4.432633
1589,3567,3.486414,5.000000,5.0,4.394565
3602,2543,5.130034,1.513039,5.0,4.354621
1424,144,3.303205,5.000000,5.0,4.321282
3084,2342,3.676360,4.246514,5.0,4.319847
750,203,3.227051,5.000000,5.0,4.290820


<h3><b>Predict the rating of Top-N movies rating using Meta-Level Model </b></h3>

In [31]:
def meta_predicton(metaModel, topN_df, ret_n_movie):
    metaModel_pred = metaModel.predict(topN_df[["TfIdf_Rating", "SVD_Rating"]])
    print(type(metaModel_pred))

    topN_df["Meta_Model_Rating"] = metaModel_pred
    topN_df = topN_df.sort_values(["Meta_Model_Rating"], ascending=False)
    return topN_df.iloc[0:ret_n_movie, :]


meta_predicton(gradient_model, topN_unrated_movies, 10)

<class 'numpy.ndarray'>


,MovieID,SVD_Rating,TfIdf_Rating,Pure_TfIdf_Rating,Combined_Ratinng,Meta_Model_Rating
1458,3028,5.117665,0.000000,5.0,4.047066,4.998278
2765,1746,5.192100,0.335982,5.0,4.144037,4.976489
2979,3239,5.013230,0.409718,5.0,4.087236,4.976139
3602,2543,5.130034,1.513039,5.0,4.354621,4.973391
3517,3242,4.912373,5.000000,5.0,4.964949,4.971512
1480,1235,4.875217,1.282429,5.0,4.206573,4.968900
2670,3715,4.934571,0.578961,5.0,4.089620,4.966411
2024,664,4.894151,1.513039,5.0,4.260268,4.965759
3088,212,4.872494,1.513039,5.0,4.251606,4.964374
539,1257,4.805035,0.588906,5.0,4.039795,4.956434


<h3><b> Save SVD and gradient_model </b></h3>

In [32]:
joblib.dump(gradient_model, 'gradient_model.pkl')
joblib.dump(model, 'svd_model.pkl')

['svd_model.pkl']

### save the trainset for svd-model

In [33]:
joblib.dump(svd_reader_trainset, "svd_reader_trainset.pkl")

['svd_reader_trainset.pkl']

In [34]:
# now load the model and get some prediction 
loaded_svd_model = joblib.load('svd_model.pkl')
loaded_gradient_model = joblib.load('gradient_model.pkl')

In [35]:
print(loaded_svd_model.pu)
print(loaded_svd_model.qi)
print()
print()

gradDF = pd.DataFrame({
    "TfIdf_Rating": [5.0], 
    "SVD_Rating": [4.392598]	
})
print(loaded_gradient_model.predict(gradDF[["TfIdf_Rating", "SVD_Rating"]]))

[[ 0.02605203  0.14382237 -0.05705039 ...  0.16767079  0.06834559
   0.10887697]
 [ 0.00588623 -0.10596437 -0.162155   ...  0.18862373  0.06708344
  -0.08342314]
 [ 0.16067624 -0.03050688  0.08916809 ...  0.21558999  0.00649929
   0.1094536 ]
 ...
 [ 0.19718196  0.33171952 -0.00196104 ...  0.24012333  0.01288041
   0.10796696]
 [-0.29943481  0.10608914 -0.13318186 ... -0.1147288   0.10819077
   0.23798154]
 [-0.09506019  0.16384504  0.07931669 ... -0.15584927 -0.20226737
  -0.16003078]]
[[-0.32006605 -0.00435392 -0.24182974 ... -0.21537233 -0.22238225
  -0.06333937]
 [-0.04574408 -0.15102971 -0.01905096 ...  0.0210476  -0.1230412
   0.11665338]
 [-0.06404831  0.14997721 -0.14941594 ...  0.37181397 -0.02585439
   0.07552321]
 ...
 [-0.01634214 -0.05143964  0.08071003 ...  0.25878392  0.01753499
   0.0819853 ]
 [ 0.10546717  0.01977039  0.04207478 ...  0.01896833  0.07939374
   0.04708637]
 [-0.23037158  0.13913113  0.11923325 ...  0.10969583 -0.10966762
   0.04381581]]


[4.57988167]
